In [1]:
#%pip install pandas==2.2.3
#%pip install git+https://github.com/pydata/pandas-datareader.git
#%pip install yfinance numpy matplotlib

In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error
import joblib

In [3]:
df = pd.read_csv('../../dataset_completo.csv')
df.head()

df.drop(columns=['Date'], inplace=True) #Eliminamos la columna 'Date' ya que no es una característica numérica y no aporta información relevante para el modelo. Además, eliminamos la columna 'target' ya que es la variable que queremos predecir.

In [4]:
X = df.drop(columns=['target']) #Eliminamos la columna 'Date' ya que no es una característica numérica y no aporta información relevante para el modelo. Además, eliminamos la columna 'target' ya que es la variable que queremos predecir.
Y = df['target']

In [5]:
# Función para generar las particiones preservando las características
# de la serie de tiempo

def division_conjuntos(dataframe, tr_size=0.8, vl_size=0.1, ts_size=0.1 ):
    # Definir número de datos en cada subserie
    N = dataframe.shape[0]
    Ntrain = int(tr_size*N)  # Número de datos de entrenamiento
    Nval = int(vl_size*N)    # Número de datos de validación
    Ntst = N - Ntrain - Nval # Número de datos de prueba

    # Realizar partición
    train = dataframe[0:Ntrain]
    val = dataframe[Ntrain:Ntrain+Nval]
    test = dataframe[Ntrain+Nval:]

    return train, val, test

# Prueba de la función
tr, vl, ts = division_conjuntos(df)

X_train = tr.drop(columns=['target'])
y_train = tr['target']

X_val = vl.drop(columns=['target'])
y_val = vl['target']

X_test = ts.drop(columns=['target'])
y_test = ts['target']

print(f'Entrenamiento: {X_train.shape}')
print(f'Validación: {X_val.shape}')
print(f'Prueba: {X_test.shape}')

Entrenamiento: (212, 26)
Validación: (26, 26)
Prueba: (27, 26)


In [6]:
# ============================================
# GRADIENT BOOSTING - BÚSQUEDA DE HIPERPARÁMETROS
# ============================================

# Definir el espacio de búsqueda
param_grid_gb = {
    'n_estimators': [100, 150, 200, 250, 300],
    'learning_rate': [0.005, 0.01, 0.03, 0.05, 0.1],
    'max_depth': [3, 5, 7, 9],
    'subsample': [0.4, 0.6, 0.8, 1.0],
    'min_samples_split': [2, 5, 8, 10, 12, 15]
}

# Validación cruzada temporal (misma que para Random Forest)
ts_cv = TimeSeriesSplit(n_splits=3)

# Inicializar el modelo
gb = GradientBoostingRegressor(random_state=42)

# Búsqueda en rejilla
grid_search_gb = GridSearchCV(
    estimator=gb,
    param_grid=param_grid_gb,
    cv=ts_cv,
    scoring='neg_mean_squared_error',
    verbose=1,
    n_jobs=-1
)

# Ajustar la búsqueda
grid_search_gb.fit(X_train, y_train)

# Mejores hiperparámetros
print("\n=== GRADIENT BOOSTING ===")
print(f"Mejores hiperparámetros: {grid_search_gb.best_params_}")
print(f"Mejor MSE: {-grid_search_gb.best_score_:.4f}")

Fitting 3 folds for each of 2400 candidates, totalling 7200 fits

=== GRADIENT BOOSTING ===
Mejores hiperparámetros: {'learning_rate': 0.005, 'max_depth': 7, 'min_samples_split': 15, 'n_estimators': 100, 'subsample': 0.4}
Mejor MSE: 0.0074


In [7]:
# ============================================
# GRADIENT BOOSTING - ENTRENAMIENTO FINAL
# ============================================

# Entrenar el modelo final con los mejores hiperparámetros
best_gb = GradientBoostingRegressor(**grid_search_gb.best_params_, random_state=42)
best_gb.fit(X_train, y_train)

# Predicciones sobre validación
y_pred_val = best_gb.predict(X_val)

# Métricas de validación
mae_val = mean_absolute_error(y_val, y_pred_val)
rmse_val = np.sqrt(mean_squared_error(y_val, y_pred_val))

print(f"\n=== GRADIENT BOOSTING - VALIDACIÓN ===")
print(f"MAE: {mae_val:.4f}")
print(f"RMSE: {rmse_val:.4f}")


=== GRADIENT BOOSTING - VALIDACIÓN ===
MAE: 0.0723
RMSE: 0.0843


=== GRADIENT BOOSTING - VALIDACIÓN ===
MAE: 0.0716
RMSE: 0.0834



In [8]:
joblib.dump(best_gb, 'gradient_boosting_model.pkl')
print("\nModelo Gradient Boosting guardado como 'gradient_boosting_model.pkl'")


Modelo Gradient Boosting guardado como 'gradient_boosting_model.pkl'
